# os module
- The Python `os` module provides tools for using operating system-dependent functionality, like reading or writing to the file system. It allows you to interface with the underlying operating system in a portable way.
- Most used parts: **current directory**, **environment variables**, **listing / creating / deleting files & folders**, and **`os.path`** for working with path strings.

In [1]:
import os

### 1. System info & current directory
| Function | What it does |
|---|---|
| `os.name` | OS type: `'posix'` (Linux/macOS) or `'nt'` (Windows) |
| `os.getcwd()` | current working directory |
| `os.chdir(path)` | change working directory |
| `os.sep` | path separator: `'/'` on Linux, `'\\'` on Windows |
| `os.cpu_count()` | number of CPU cores (useful for `n_jobs` / `num_workers` in ML) |

In [2]:
print(os.name)
print(os.getcwd())
print(os.sep)
print(os.cpu_count())

posix
/home/mahmoudnasr/Desktop/ml-journey/01-python/std-lib-notes
/
12


### 2. Environment variables
- `os.environ` is a dict-like object of all environment variables.
- Prefer `os.getenv("KEY", default)` — it returns `None`/default instead of raising `KeyError`.
- Very common for API keys and config: never hard-code secrets, read them from the environment.

In [3]:
print(os.environ["HOME"])                 # KeyError if missing
print(os.getenv("HOME"))                  # None if missing
print(os.getenv("MY_API_KEY", "not-set")) # with a default value

os.environ["MY_VAR"] = "hello"            # set (only for this process)
print(os.getenv("MY_VAR"))

/home/mahmoudnasr
/home/mahmoudnasr
not-set
hello


### 3. Creating & deleting directories
| Function | What it does |
|---|---|
| `os.mkdir(path)` | create **one** directory (error if exists or parent missing) |
| `os.makedirs(path, exist_ok=True)` | create nested dirs (`a/b/c`), no error if it already exists ⭐ |
| `os.rmdir(path)` | remove an **empty** directory |
| `shutil.rmtree(path)` | remove a directory **with all its content** (from `shutil`, careful!) |

In [4]:
os.mkdir("playground")                               # create dir
os.makedirs("playground/data/raw", exist_ok=True)    # nested dirs, safe to re-run
print(os.listdir("playground"))

['data']


### 4. Listing directory content
- `os.listdir(path)` → list of names (files + folders), no full path.
- `os.scandir(path)` → iterator of `DirEntry` objects (has `.name`, `.path`, `.is_file()`, `.is_dir()`), faster than listdir + checks.
- `os.walk(path)` → walks the **whole tree**; yields `(dirpath, dirnames, filenames)` for every folder. It's a generator, so loop over it (just calling `os.walk(...)` prints nothing useful).

In [5]:
# create some files to play with
for name in ["a.txt", "b.csv", "data/raw/c.txt"]:
    with open(os.path.join("playground", name), "w") as f:
        f.write("hello")

print(os.listdir("playground"))

['a.txt', 'data', 'b.csv']


In [7]:
for entry in os.scandir("playground"):
    print(entry.name, "-> file" if entry.is_file() else "-> dir")

a.txt -> file
data -> dir
b.csv -> file


In [8]:
for dirpath, dirnames, filenames in os.walk("playground"):
    print(dirpath, dirnames, filenames)

playground ['data'] ['a.txt', 'b.csv']
playground/data ['raw'] []
playground/data/raw [] ['c.txt']


In [9]:
# common pattern: collect all .txt files recursively
txt_files = []
for dirpath, _, filenames in os.walk("playground"):
    for f in filenames:
        if f.endswith(".txt"):
            txt_files.append(os.path.join(dirpath, f))
txt_files

['playground/a.txt', 'playground/data/raw/c.txt']

### 5. Files: rename, remove, info
| Function | What it does |
|---|---|
| `os.rename(src, dst)` | rename / move file or dir |
| `os.replace(src, dst)` | like rename but overwrites `dst` if it exists |
| `os.remove(path)` | delete a **file** (not a dir) |
| `os.stat(path)` | file info: `.st_size` (bytes), `.st_mtime` (last modified) |

In [10]:
os.rename("playground/a.txt", "playground/a_renamed.txt")
print(os.listdir("playground"))

print(os.stat("playground/b.csv").st_size, "bytes")

os.remove("playground/b.csv")
print(os.listdir("playground"))

['data', 'a_renamed.txt', 'b.csv']
5 bytes
['data', 'a_renamed.txt']


### 6. `os.path` — working with path strings ⭐
Paths here are plain **strings**. Always use `os.path.join` instead of `"folder" + "/" + "file"` so the code works on every OS.

| Function | Example result for `"playground/data/raw/c.txt"` |
|---|---|
| `os.path.join(a, b, ...)` | build a path |
| `os.path.exists(p)` | `True` / `False` |
| `os.path.isfile(p)` / `os.path.isdir(p)` | type checks |
| `os.path.basename(p)` | `'c.txt'` |
| `os.path.dirname(p)` | `'playground/data/raw'` |
| `os.path.split(p)` | `('playground/data/raw', 'c.txt')` |
| `os.path.splitext(p)` | `('playground/data/raw/c', '.txt')` |
| `os.path.abspath(p)` | full absolute path |
| `os.path.getsize(p)` | size in bytes |
| `os.path.expanduser("~")` | home directory |

In [11]:
p = os.path.join("playground", "data", "raw", "c.txt")
print(p)
print(os.path.exists(p), os.path.isfile(p), os.path.isdir(p))
print(os.path.basename(p))
print(os.path.dirname(p))
print(os.path.split(p))
print(os.path.splitext(p))
print(os.path.abspath(p))
print(os.path.getsize(p))
print(os.path.expanduser("~"))

playground/data/raw/c.txt
True True False
c.txt
playground/data/raw
('playground/data/raw', 'c.txt')
('playground/data/raw/c', '.txt')
/home/mahmoudnasr/Desktop/ml-journey/01-python/std-lib-notes/playground/data/raw/c.txt
5
/home/mahmoudnasr


In [12]:
# common pattern: only create a file/dir if it doesn't exist
if not os.path.exists("playground/models"):
    os.mkdir("playground/models")
print(os.listdir("playground"))

['models', 'data', 'a_renamed.txt']


In [13]:
# cleanup
import shutil
shutil.rmtree("playground")
print(os.path.exists("playground"))

False


## 🏋️ Exercises
- Try each exercise yourself in the empty cell first, then open **💡 Show solution** to compare.
- Run the **setup** cell first — it creates a practice folder with sample files.
- If something breaks, just re-run the setup cell to reset everything.

In [2]:
# SETUP: creates a practice folder (re-run anytime to reset)
import os, shutil

shutil.rmtree("ex_os", ignore_errors=True)
sample = {
    "ex_os/notes.txt": "hello",
    "ex_os/todo.txt": "learn os module",
    "ex_os/data/train.csv": "x,y\n" + "1,2\n" * 100,
    "ex_os/data/test.csv": "x,y\n1,2\n",
    "ex_os/data/images/cat.jpg": "fake image" * 50,
    "ex_os/data/images/dog.jpg": "fake image",
    "ex_os/logs/run1.log": "epoch 1 done",
}
for path, content in sample.items():
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        f.write(content)
print("setup done:", os.listdir("ex_os"))

setup done: ['notes.txt', 'data', 'logs', 'todo.txt']


### Exercise 1
**System info** — print:
1. the current working directory
2. the number of CPU cores
3. `"Linux/macOS"` if `os.name` is `'posix'`, otherwise `"Windows"`

In [16]:
print(os.getcwd())
print(os.cpu_count())
if os.name == "posix":
    print("Linux/macOs")
else:
    print("Windows")

/home/mahmoudnasr/Desktop/ml-journey/01-python/std-lib-notes
12
Linux/macOs


<details>
<summary>💡 Show solution</summary>

```python
print(os.getcwd())
print(os.cpu_count())
print("Linux/macOS" if os.name == "posix" else "Windows")
```
</details>

### Exercise 2
**Environment variables** — read the variable `DATA_DIR`. If it isn't set, use `"./data"` as default. Print the result.
Then set `DATA_DIR` to `"ex_os/data"` and read it again.

In [19]:
# your code here
print(os.getenv("DATA_DIR", "./data"))
os.environ["DATA_DIR"] = "ex_os/data"
print(os.getenv("DATA_DIR", "./data"))

ex_os/data
ex_os/data


<details>
<summary>💡 Show solution</summary>

```python
print(os.getenv("DATA_DIR", "./data"))
os.environ["DATA_DIR"] = "ex_os/data"
print(os.getenv("DATA_DIR", "./data"))
```
</details>

### Exercise 3
**Project structure** — inside `ex_os/` create this structure with as few calls as possible.
Your code must not crash if you run it twice.
```
ex_os/project/
├── data/raw
├── data/processed
├── models
└── notebooks
```
Hint: `os.makedirs` + `exist_ok`

In [22]:
# your code here
os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)
os.makedirs("models", exist_ok=True)
os.makedirs("notebooks", exist_ok=True)


<details>
<summary>💡 Show solution</summary>

```python
for folder in ["data/raw", "data/processed", "models", "notebooks"]:
    os.makedirs(os.path.join("ex_os", "project", folder), exist_ok=True)
print(os.listdir("ex_os/project"))
```
</details>

### Exercise 4
**Check before you act** — write a function `safe_remove(path)` that:
- deletes the path if it's a **file** and prints `"removed file"`
- prints `"is a directory, skipped"` if it's a directory
- prints `"not found"` if it doesn't exist

Test it on `ex_os/notes.txt`, `ex_os/data`, and `ex_os/nothing.txt`.

In [3]:
# your code here
def safe_remove(path):
    if os.path.isfile(path):
        os.remove(path)
        print("removed file")
    elif os.path.isdir(path):
        print("is a directory, skipped")
    else:
        print("not found")
safe_remove("ex_os/notes.txt")
safe_remove("ex_os/data")
safe_remove("ex_os/nothing.txt")


removed file
is a directory, skipped
not found


<details>
<summary>💡 Show solution</summary>

```python
def safe_remove(path):
    if os.path.isfile(path):
        os.remove(path)
        print("removed file")
    elif os.path.isdir(path):
        print("is a directory, skipped")
    else:
        print("not found")

safe_remove("ex_os/notes.txt")
safe_remove("ex_os/data")
safe_remove("ex_os/nothing.txt")
```
</details>

### Exercise 5
**Count files by extension** — use `os.walk` to go through all of `ex_os/` and build a dict like `{'.csv': 2, '.jpg': 2, ...}`.
Hint: `os.path.splitext`

In [4]:
counts = {}
for dirpath, dirnames, filenames in os.walk("ex_os"):
    for name in filenames:
        ext = os.path.splitext(name)[1]
        counts[ext] = counts.get(ext, 0) + 1
print(counts)

{'.txt': 1, '.csv': 2, '.jpg': 2, '.log': 1}


<details>
<summary>💡 Show solution</summary>

```python
counts = {}
for dirpath, dirnames, filenames in os.walk("ex_os"):
    for name in filenames:
        ext = os.path.splitext(name)[1]
        counts[ext] = counts.get(ext, 0) + 1
print(counts)
```
</details>

### Exercise 6
**Find the largest file** — find the biggest file anywhere inside `ex_os/` and print its full path and size in bytes.
Hint: `os.walk` + `os.path.getsize`

In [6]:
fullpath,max_size = None,-1
for dirpath, _, filenames in os.walk("ex_os"):
    for name in filenames:
        full = os.path.join(dirpath, name)
        size = os.path.getsize(full)
        if size > max_size:
            fullpath, max_size = full, size

print(fullpath)
print(max_size)

ex_os/data/images/cat.jpg
500


<details>
<summary>💡 Show solution</summary>

```python
largest_path, largest_size = None, -1
for dirpath, _, filenames in os.walk("ex_os"):
    for name in filenames:
        full = os.path.join(dirpath, name)
        size = os.path.getsize(full)
        if size > largest_size:
            largest_path, largest_size = full, size
print(largest_path, largest_size, "bytes")
```
</details>

### Exercise 7
**Batch rename** — rename every `.txt` file directly inside `ex_os/` to `.md` (e.g. `todo.txt` → `todo.md`). Print the folder content after.

In [10]:
for dirpath, dirnames, filenames in os.walk("ex_os"):
    for name in filenames:
        root, ext = os.path.splitext(name)
        if ext == ".txt":
            os.rename(os.path.join("ex_os", name), os.path.join("ex_os", root + '.md'))


<details>
<summary>💡 Show solution</summary>

```python
for name in os.listdir("ex_os"):
    root, ext = os.path.splitext(name)
    if ext == ".txt":
        os.rename(os.path.join("ex_os", name), os.path.join("ex_os", root + ".md"))
print(os.listdir("ex_os"))
```
</details>

### Cleanup
Run this when you're done to delete the practice folder.

In [11]:
shutil.rmtree("ex_os", ignore_errors=True)
print(os.path.exists("ex_os"))

False
